<a href="https://colab.research.google.com/github/vtminc1000/Analisis-Sentimen/blob/main/IndoBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas openpyxl transformers torch scikit-learn -q

In [ ]:
import pandas as pd
import re
from google.colab import files
from transformers import pipeline

In [ ]:
uploaded = files.upload()

In [ ]:
df = pd.read_excel('Data_Inite.xlsx')
df.head()

In [ ]:
data_long = []

for _, row in df.iterrows():
    data_long.append({'responden_id': row['responden_id'], 'tahap': 'pre', 'pertanyaan': 'q1', 'teks': row['pre_q1'], 'label_asli': row['pre_q1_label']})
    data_long.append({'responden_id': row['responden_id'], 'tahap': 'pre', 'pertanyaan': 'q2', 'teks': row['pre_q2'], 'label_asli': row['pre_q2_label']})
    data_long.append({'responden_id': row['responden_id'], 'tahap': 'post', 'pertanyaan': 'q1', 'teks': row['post_q1'], 'label_asli': row['post_q1_label']})
    data_long.append({'responden_id': row['responden_id'], 'tahap': 'post', 'pertanyaan': 'q2', 'teks': row['post_q2'], 'label_asli': row['post_q2_label']})

data_long = pd.DataFrame(data_long)
data_long.head()

In [ ]:
print(data_long.shape)

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data_long['teks_clean'] = data_long['teks'].apply(clean_text)
data_long[['teks', 'teks_clean']].head()

In [ ]:
classifier = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

In [ ]:
def predict_sentiment(text):
    result = classifier(str(text))[0]
    label = result['label']

    if label == 'negative':
        return -1
    elif label == 'neutral':
        return 0
    else:
        return 1

In [ ]:
print(predict_sentiment("saya sangat senang"))
print(predict_sentiment("biasa saja"))
print(predict_sentiment("ini berbahaya"))

In [ ]:
data_long['indobert_label'] = data_long['teks_clean'].apply(predict_sentiment)

In [ ]:
data_long[['teks', 'label_asli', 'indobert_label']].head(15)

In [ ]:
data_long[['teks', 'label_asli', 'indobert_label']].head(15)

In [ ]:
data_q1 = data_long[data_long['pertanyaan'] == 'q1']

distribusi_q1 = data_q1.groupby(['tahap', 'indobert_label']).size().unstack(fill_value=0)

distribusi_q1 = distribusi_q1.rename(columns={
    -1: 'Negatif',
    0: 'Netral',
    1: 'Positif'
})

distribusi_q1

In [ ]:
data_q2 = data_long[data_long['pertanyaan'] == 'q2']

distribusi_q2 = data_q2.groupby(['tahap', 'indobert_label']).size().unstack(fill_value=0)

distribusi_q2 = distribusi_q2.rename(columns={
    -1: 'Negatif',
    0: 'Netral',
    1: 'Positif'
})

distribusi_q2

In [ ]:
distribusi_q1.to_excel('distribusi_q1_indobert.xlsx')
distribusi_q2.to_excel('distribusi_q2_indobert.xlsx')

data_long.to_excel('hasil_indobert.xlsx', index=False)

from google.colab import files
files.download('distribusi_q1_indobert.xlsx')
files.download('distribusi_q2_indobert.xlsx')
files.download('hasil_indobert.xlsx')

In [ ]:
data_long[['teks', 'label_asli', 'indobert_label']].head(10)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
distribusi_q1.plot(kind='bar')

plt.title('Distribusi Sentimen Q1 (IndoBERT)')
plt.xlabel('Tahap')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
plt.legend(title='Sentimen')

plt.show()

In [ ]:
distribusi_q2.plot(kind='bar')

plt.title('Distribusi Sentimen Q2 (IndoBERT)')
plt.xlabel('Tahap')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
plt.legend(title='Sentimen')

plt.show()

In [ ]:
distribusi_q1_persen = distribusi_q1.div(distribusi_q1.sum(axis=1), axis=0) * 100

distribusi_q1_persen.plot(kind='bar')

plt.title('Distribusi Sentimen (%) Q1 (IndoBERT)')
plt.xlabel('Tahap')
plt.ylabel('Persentase (%)')
plt.xticks(rotation=0)

plt.show()

In [ ]:
distribusi_q2_persen = distribusi_q2.div(distribusi_q2.sum(axis=1), axis=0) * 100

distribusi_q2_persen.plot(kind='bar')

plt.title('Distribusi Sentimen (%) Q2 (IndoBERT)')
plt.xlabel('Tahap')
plt.ylabel('Persentase (%)')
plt.xticks(rotation=0)

plt.show()

In [ ]:
plt.figure()
distribusi_q1.plot(kind='bar')
plt.title('Distribusi Sentimen Q1 (IndoBERT)')
plt.xlabel('Tahap')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)

plt.savefig('grafik_q1_indobert.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure()
distribusi_q2.plot(kind='bar')
plt.title('Distribusi Sentimen Q2 (IndoBERT)')
plt.xlabel('Tahap')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)

plt.savefig('grafik_q2_indobert.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from google.colab import files

files.download('grafik_q1_indobert.png')
files.download('grafik_q2_indobert.png')